In [1]:
import time
import json
import requests
import re

import pandas as pd
pd.set_option('display.max_columns', None)

from tqdm.notebook import tqdm_notebook

PRODUCT_URL = "https://world.openfoodfacts.org/api/v2/product"

In [2]:
df_mups = pd.read_excel('MUP.csv')
df_mups.head()

,category,mup_id,mup_name,individual_compound,match_type,is_extended
0,cosmetic_additives__flavours,1,flavour,flavor,exact,False
1,cosmetic_additives__flavours,1,flavour,flavour,exact,False
2,cosmetic_additives__flavour_enhancers,2,e62*,e620 - e629,prefix,False
3,cosmetic_additives__flavour_enhancers,3,e63*,e630 - e637,prefix,False
4,cosmetic_additives__flavour_enhancers,4,e640,e640,exact,False


## Section 1: Inspect Scraped Data

### **1.1 Inspect Waitrose Data** 

In [2]:
df_waitrose = pd.read_json('/files/problem-set-1-grayusan/data/scraped/products.jsonl', lines=True)
df_waitrose.head()

,name,barcode,url,food_type,pricing,ingredients
0,GOOD TO GO Pomegranate Seeds,"[5000169631362, 5000169158333]",https://www.waitrose.com/ecom/products/good-to...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",1.65,INGREDIENTS: pomegranate seeds (100%).
1,Duchy Organic Smoked Salmon,[5000169173701],https://www.waitrose.com/ecom/products/duchy-o...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",5.62,INGREDIENTS: Organic salmon (Salmo salar) (fis...
2,Natures Candy USA Medjool Dates,[5061048250005],https://www.waitrose.com/ecom/products/natures...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",5.25,None
3,Essential Mild Cheddar S2,[5000169577851],https://www.waitrose.com/ecom/products/essenti...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",2.75,INGREDIENTS: Cheddar Cheese (milk).
4,No.1 Keralan Inspired Fish Curry for 2,[5063210008216],https://www.waitrose.com/ecom/products/no1-ker...,"[{'id': '10051', 'name': 'Groceries'}, {'id': ...",6.50,"INGREDIENTS: Hake (fish) (52%), onion, coconut..."


In [ ]:
for product in df_waitrose['food_type']: 
    df_waitrose['food_type'][0]

### **1.2 OpenFoodFacts responses**

In [9]:
openfoodfacts_responses = []

sample = df_waitrose['barcode'].head(5)

for idx, barcodes in tqdm_notebook(sample.items()):
    row_results = []
    for barcode in barcodes:
        response = requests.get(f'{PRODUCT_URL}/{barcode}', timeout=30)
        if response.status_code == 200:
            result = response.json()
            row_results.append({
                'product_id': idx,
                'barcode_used': barcode,
                'response': result,
            })
        time.sleep(0.5)

    if not row_results:
        openfoodfacts_responses.append({
            'product_id': idx,
            'barcode_used': barcodes[0] if barcodes else None,
            'response': None,
        })

    else:
        openfoodfacts_responses.extend(row_results)

with open("/files/problem-set-1-grayusan/api/off_data_validation_sample.json", "w") as f:
    json.dump(openfoodfacts_responses, f, indent=2, ensure_ascii=False)
print(f"Saved {len(openfoodfacts_responses)} responses to off_data_validation_sample.json")

0it [00:00, ?it/s]

Saved 5 responses to off_data_validation_sample.json


In [3]:
with open('off_data_validation_sample.json', 'r') as f:
    api_data = json.load(f)

In [ ]:
df_api = pd.json_normalize(api_data, meta=['response', 'product'])
df_api.columns.to_list()

In [8]:
df_api['response.product.nova_groups_tags']

0          NaN
1    [unknown]
2          NaN
3    [unknown]
4          NaN
Name: response.product.nova_groups_tags, dtype: object